# EFT ISY0101 - Agente Multi-Especialista Canon iX6810
## Imprenta Nueva Imagen

Notebook de entrega para la Evaluación Final Transversal.
Cubre: RAG híbrido, arquitectura multi-agente, memoria, planificación, observabilidad y seguridad.

## 1. Instalación de dependencias

In [ ]:
!pip install -q langchain langchain-community langchain-openai faiss-cpu pypdf python-dotenv openai streamlit plotly pandas

## 2. Configuración de entorno
Defina `OPENAI_API_KEY` en Colab: Entorno de ejecución > Variables de entorno.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
assert os.getenv('OPENAI_API_KEY') or os.getenv('GITHUB_TOKEN'), 'Configure OPENAI_API_KEY'

## 3. Pipeline RAG - Fuentes internas y externas (IE2)

In [ ]:
from imprenta.ingesta import build_faiss_index

build_faiss_index()

## 4. Construcción del agente multi-especialista (IE5, IE6, IE7)

In [ ]:
from imprenta.tasks import build_agent_executor
from imprenta.agent_monitoring import ObservableRAGAgent
from imprenta.security import sanitize_input, append_ethical_notice

coordinator = build_agent_executor()
agent = ObservableRAGAgent(coordinator, enable_metrics=True)
print('Agente listo:', type(coordinator).__name__)

## 5. Demo de consultas con memoria y planificación

In [ ]:
consultas = [
    'La Canon iX6810 imprime con líneas, ¿qué debo revisar?',
    'Genera una orden urgente: el equipo no responde y la tinta está crítica.',
    'Necesito imprimir 200 folletos A3 color, ¿qué equipo uso?',
]

for q in consultas:
    valid = sanitize_input(q)
    if not valid.allowed:
        print('Rechazada:', valid.reason)
        continue
    resp = agent.ask(valid.sanitized_query)
    print('=' * 60)
    print('Consulta:', q)
    print(coordinator.get_orchestration_summary())
    print('Respuesta:', append_ethical_notice(resp)[:500], '...')

## 6. Observabilidad - Métricas y trazabilidad (IE9, IE10)

In [ ]:
from imprenta.test_scenarios import main as run_tests
from imprenta.observability import get_metrics_collector

run_tests()
print(get_metrics_collector().get_summary_metrics())

## 7. Dashboard de métricas
Ejecutar en terminal local: `streamlit run imprenta/dashboard.py`

## 8. Demo local sin API (reproducible)
Alternativa cuando no hay claves configuradas:

In [ ]:
!python -m imprenta.demo_local